In [1]:
import sys, hashlib, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))
from sneak.fcg_registry import FCGRegistry
from sneak.merkle import canonical
from replay_sneak_receipt_fcg import verify_source
print('SOURCE=', ROOT / 'receipts/connectors/SNEAK-016_LOMO_SNYK_SCORE.json')

SOURCE= /Users/byron/projects/hackathon/sneak-audit-fco-20261020/receipts/connectors/SNEAK-016_LOMO_SNYK_SCORE.json


In [2]:
source = ROOT / 'receipts/connectors/SNEAK-016_LOMO_SNYK_SCORE.json'
raw = source.read_bytes()
receipt = json.loads(raw)
verify_source(receipt)
print('SOURCE_MERKLE_PASS', receipt['root'], hashlib.sha256(raw).hexdigest())

SOURCE_MERKLE_PASS 72e7a22ae6e2e46be6b09f60af3f5efb13603fd84a4dee08b8706fd11543f6ae f8de91405ee4d924bfbd13703477f63b817a3e5957eae984b6526264dfe46b4b


In [3]:
ledger = FCGRegistry(ROOT / 'data/fcg/sneak-016-replay.jsonl')
result = ledger.verify()
proof = ledger.inclusion_proof(ledger.rows()[-1]['fco_id'])
assert proof['verified']
assert result['fco_count'] == 55 and result['mmr_size'] == 105 and result['fcg_edges'] == 64
print(json.dumps(result, indent=2, sort_keys=True))

{
  "fcg_edges": 64,
  "fco_count": 55,
  "mmr_root": "089e49c083f1d6170402a366cfa76f3b939cbdcefce262ff39a732061463dadc",
  "mmr_size": 105,
  "verified": true
}


In [4]:
assert receipt['mmr'] == 'NOT_COMMITTED'
assert sum(r['identity']['kind'] == 'scan' and r['identity']['state'] == 'NOT_TESTED' for r in ledger.rows()) == 8
assert sum(r['identity']['kind'] == 'score' and r['identity']['state'] == 'NULL' for r in ledger.rows()) == 8
assert sum(r['identity']['kind'] == 'delta' for r in ledger.rows()) == 4
print('PASS: source Merkle verified, successor MMR verified, scan failures preserved')

PASS: source Merkle verified, successor MMR verified, scan failures preserved
